<a href="https://colab.research.google.com/github/hadi-hosseini/bandit/blob/main/Bandit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [518]:
import math
import cvxpy as cp
import numpy as np
from scipy.stats import ortho_group
from tqdm import tqdm

# Parameters
k = 4
d = 1000
n_samples = 1000
sigma = 1.0
T = 500

orthogonal_matrix = ortho_group.rvs(dim=d)
mu = orthogonal_matrix[:k]
print(mu)

[[ 0.03334054  0.01852866 -0.0688507  ... -0.02637434  0.06350634
  -0.0100525 ]
 [-0.01363877 -0.03709595  0.03575663 ...  0.01988552 -0.03828403
   0.00803181]
 [ 0.0204218  -0.02249485  0.02422479 ...  0.00025926  0.0312108
   0.02526309]
 [ 0.00243777 -0.06619986 -0.04309034 ...  0.03224677 -0.01883651
  -0.02148103]]


In [519]:
# Verify orthogonality
for i in range(k):
    for j in range(i+1, k):
        print(f"Dot product μ_{i+1}·μ_{j+1}: {np.dot(mu[i], mu[j])}")

Dot product μ_1·μ_2: -1.1614515772750966e-17
Dot product μ_1·μ_3: -8.586881206085195e-17
Dot product μ_1·μ_4: 7.26415455565288e-18
Dot product μ_2·μ_3: 3.7025504190379976e-17
Dot product μ_2·μ_4: 9.679214894864341e-17
Dot product μ_3·μ_4: -4.651227319962814e-17


In [520]:
# create offline dataset
def create_logged_data(k, d, n_samples, sigma, mu):
  samples = []
  for i in range(k):
    # Generate samples from N(μ_i, σ²I)
    samples.append(np.random.normal(loc=mu[i], scale=sigma, size=(n_samples, d)))
  return samples

logged_data = create_logged_data(k, d, n_samples, sigma, mu)

In [521]:
# implement UCB algorithm
class UCBAlgorithm:
    def __init__(self, k, d, true_means, logged_data, perturbation):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data

        self.N = np.zeros(k)
        self.total_rewards = np.zeros(k)
        self.empirical_rewards = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = perturbation

    def get_reward(self, x):
        return np.dot(self.true_means[0] + self.perturbation, x)

    def select_arm(self, t):
        if t < self.k:
            return t

        ucb_values = np.zeros(self.k)
        for j in range(self.k):
            mean_term = self.empirical_rewards[j]
            confidence_bound = math.sqrt((2 * math.log(t)) / self.N[j])

            ucb_values[j] = mean_term + confidence_bound

        return np.argmax(ucb_values)

    def update(self, arm, reward):
        self.N[arm] += 1
        self.total_rewards[arm] += reward
        self.empirical_rewards[arm] = self.total_rewards[arm] / self.N[arm]

    def run(self, T):
        rewards = np.zeros(T)
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)
            sample = self.logged_data[arm][int(self.N[arm])]
            reward = self.get_reward(sample)

            self.update(arm, reward)
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            rewards[t] = reward
            chosen_arms[t] = arm

        return rewards, chosen_arms

In [522]:
# run UCB without perturbation
perturbation = 0.0
ucb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb.run(T)
print("\nNumber of pulls per arm:", ucb.N)
print(chosen_arms)

100%|██████████| 500/500 [00:00<00:00, 34675.71it/s]


Number of pulls per arm: [469.  14.   8.   9.]
[0 1 2 3 0 3 0 1 0 0 0 0 0 0 0 0 0 0 2 3 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
 0 0 0 1 0 0 0 0 0 2 2 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 2 1 3 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 2 0 0 0 0 0 0 0 3 0 1 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 2 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [523]:
def random_perturbation(d, epsilon):
    perturbation = np.random.randn(d)
    perturbation = epsilon * perturbation / np.linalg.norm(perturbation)
    return perturbation

epsilon = 0.5
random_perturbation = random_perturbation(d, epsilon)
ucb = UCBAlgorithm(k, d, mu, logged_data, random_perturbation)
rewards, chosen_arms = ucb.run(T)
print("\nNumber of pulls per arm:", ucb.N)
print(chosen_arms)

100%|██████████| 500/500 [00:00<00:00, 45491.37it/s]


Number of pulls per arm: [468.  14.   9.   9.]
[0 1 2 3 3 1 0 0 0 0 0 0 0 0 3 2 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 2 2 0 0 0 1
 0 1 0 0 0 0 0 0 0 0 0 0 0 1 3 0 0 0 0 1 0 0 0 0 0 2 0 0 0 0 1 3 0 0 0 0 0
 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 3 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
 0 0 0 0 0 0 0 0 3 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [524]:
# find adversary perturbation
# should learn this perturbation based on the logged data

class FindPerturbation:
    def __init__(self, k, d, true_means, logged_data, epsilon, qp=False, M=1):
        self.k = k
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data
        self.epsilon = epsilon
        self.qp = qp
        self.M = M # alternatives

        self.N = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = None
        self.history = []
        self.all_perturbs = []
        self.turn = 1

    def select_arm(self, t):
        if t < self.k:
            return t

        ### targetted
        # turn = self.turn

        ### untargetted
        self.turn += 1
        if self.turn == k:
          self.turn = 1

        return self.turn

    def find_perturbation_with_l2_ball_optimal_only(self, arm, t):
        x = cp.Variable(self.d)

        d_0 = self.empirical_means[arm] - self.empirical_means[0]
        c_0 = (math.sqrt(2 * math.log(t) / self.N[0]) - math.sqrt(2 * math.log(t) / self.N[arm])) - np.dot(self.true_means[0], d_0)
        self.history.append((d_0, c_0))


        constraints = []
        for (d_0, c_0) in self.history:
            constraints.append(x @ d_0 >= c_0 + 1e-6)
        constraints.append(cp.norm(x, 2) <= self.epsilon)
        prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          self.all_perturbs.append(x.value)
          return x.value
        else:
          return None

    def find_perturbation_with_l2_ball(self, arm, t):
        x = cp.Variable(self.d)

        for j in range(self.k):
            if j != arm:
              d_j = self.empirical_means[arm] - self.empirical_means[j]
              c_j = (math.sqrt((2 * math.log(t)) / self.N[j]) - math.sqrt((2 * math.log(t)) / self.N[arm])) - np.dot(self.true_means[0], d_j)
              self.history.append((d_j, c_j))

        constraints = []
        for (d_j, c_j) in self.history:
            constraints.append(x @ d_j >= c_j + 1e-6)
        if qp:
          objective = cp.Minimize(cp.norm(x, 2))
          prob = cp.Problem(objective, constraints)
        else:
          constraints.append(cp.norm(x, 2) <= self.epsilon)
          prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          self.all_perturbs.append(x.value)
          return x.value
        else:
          return None


    def run(self, T, mode=1):
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)

            if t >= self.k and ((t-k) % self.M == 0):
              if mode == 1: # check all inequalities
                perturbation = self.find_perturbation_with_l2_ball(arm, t)
              elif mode == 2: # check just optimal inequalities
                perturbation = self.find_perturbation_with_l2_ball_optimal_only(arm, t)


              if perturbation is None:
                return chosen_arms

              self.perturbation = perturbation

            sample = self.logged_data[arm][int(self.N[arm])]
            self.N[arm] += 1
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            chosen_arms[t] = arm

        return chosen_arms

## ABLATION 1 (CHECK ALL INEQUALITIES)

In [ ]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

 80%|███████▉  | 399/500 [1:11:58<58:01, 34.48s/it]

In [ ]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

In [ ]:
# all_perturbs = find_perturbation.all_perturbs

# for hist in all_perturbs:
#   ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, hist)
#   rewards, chosen_arms = ucb_with_perturb.run(T)
#   print("\nNumber of pulls per arm:", ucb_with_perturb.N)
#   print(chosen_arms)

In [ ]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

In [ ]:
print(len(find_perturbation.history))

## ABLATION 2 (CHECK ONLY OPTIMAL INEQUALITIES)

In [ ]:
# check just optimal inequalities (mode=2)
epsilon = 0.5
qp = False
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T, mode=2)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

In [ ]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

In [ ]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

In [ ]:
print(len(find_perturbation.history))

## ABLATION 3 (QP)

In [ ]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = True
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

In [ ]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

In [ ]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

In [ ]:
print(len(find_perturbation.history))

## ABLATION 4 (M alternatives)

In [ ]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
alternatives = True
M = 10 # alternatives
find_perturbation = FindPerturbation(k, d, mu, logged_data, epsilon, qp, M)
chosen_arms = find_perturbation.run(T, mode=1)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

In [ ]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

In [ ]:
# run UCB with perturbation
ucb_with_perturb = UCBAlgorithm(k, d, mu, logged_data, perturbation)
rewards, chosen_arms = ucb_with_perturb.run(T)
print("\nNumber of pulls per arm:", ucb_with_perturb.N)
print(chosen_arms)

In [ ]:
print(len(find_perturbation.history))

### ETC Attack

In [ ]:
# implement UCB algorithm
class ETCAlgorithm:
    def __init__(self, k, m, d, true_means, logged_data, perturbation):
        self.k = k
        self.m = m
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data

        self.N = np.zeros(k)
        self.total_rewards = np.zeros(k)
        self.empirical_rewards = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = perturbation

    def get_reward(self, x):
        return np.dot(self.true_means[0] + self.perturbation, x)

    def select_arm(self, t):
        if t < self.k * self.m:
            return t % k

        return np.argmax(self.empirical_rewards)

    def update(self, arm, reward):
        self.total_rewards[arm] += reward
        self.empirical_rewards[arm] = self.total_rewards[arm] / self.N[arm]

    def run(self, T):
        rewards = np.zeros(T)
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)
            sample = self.logged_data[arm][int(self.N[arm])]
            reward = self.get_reward(sample)
            self.N[arm] += 1

            if t < self.k * self.m:
              self.update(arm, reward)
              self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            rewards[t] = reward
            chosen_arms[t] = arm

        return rewards, chosen_arms

In [ ]:
# run ETC without perturbation
perturbation = 0.0
m = 2
etc = ETCAlgorithm(k, m, d, mu, logged_data, perturbation)
rewards, chosen_arms = etc.run(T)
print("\nNumber of pulls per arm:", etc.N)
print(chosen_arms)

In [ ]:
# find adversary perturbation
# should learn this perturbation based on the logged data

class FindPerturbationETC:
    def __init__(self, k, m, d, target_arm, true_means, logged_data, epsilon, qp=False):
        self.k = k
        self.m = m
        self.d = d
        self.true_means = true_means
        self.logged_data = logged_data
        self.epsilon = epsilon
        self.qp = qp
        self.target_arm = target_arm

        self.N = np.zeros(k)
        self.empirical_means = np.zeros((k, d))
        self.perturbation = None

    def select_arm(self, t):
        if t < self.k * self.m:
            return t % k

        return self.target_arm

    def find_perturbation_with_l2_ball(self, arm, t):
        x = cp.Variable(self.d)
        constraints = []

        for j in range(self.k):
            if j != arm:
              d_j = self.empirical_means[arm] - self.empirical_means[j]
              c_j = - np.dot(self.true_means[0], d_j)
              constraints.append(x @ d_j >= c_j + 1e-6)

        if qp:
          objective = cp.Minimize(cp.norm(x, 2))
          prob = cp.Problem(objective, constraints)
        else:
          constraints.append(cp.norm(x, 2) <= self.epsilon)
          prob = cp.Problem(cp.Minimize(0), constraints)
        prob.solve()

        if prob.status == 'optimal':
          return x.value
        else:
          return None


    def run(self, T):
        chosen_arms = np.zeros(T, dtype=int)

        for t in tqdm(range(T)):
            arm = self.select_arm(t)

            if t == self.m * self.k:
              self.perturbation = self.find_perturbation_with_l2_ball(arm, t)
              return chosen_arms

            sample = self.logged_data[arm][int(self.N[arm])]
            self.N[arm] += 1
            self.empirical_means[arm] = self.empirical_means[arm] + (sample - self.empirical_means[arm])/self.N[arm]

            chosen_arms[t] = arm

        return chosen_arms

In [ ]:
# check all inequalities (mode=1)
epsilon = 0.5
qp = False
target_arm = 2
m = 2
find_perturbation = FindPerturbationETC(k, m, d, target_arm, mu, logged_data, epsilon, qp)
chosen_arms = find_perturbation.run(T)
print("\nNumber of pulls per arm:", find_perturbation.N)
print(chosen_arms)

In [ ]:
perturbation = find_perturbation.perturbation
norm = np.linalg.norm(perturbation)
print(norm)

In [ ]:
# run ETC with perturbation
m = 2
etc = ETCAlgorithm(k, m, d, mu, logged_data, perturbation)
rewards, chosen_arms = etc.run(T)
print("\nNumber of pulls per arm:", etc.N)
print(chosen_arms)